# Chat Orchestrator Evaluation

Sanity check the orchestrator routing for chat requests.

Steps:
- Send a simple chat request.
- Capture route and response payload.
- Inspect audit log output.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from app.agent.memory import MemoryStore
from app.agent.orchestrator import SentifargoOrchestrator
from app.utils import LLMStub
from pathlib import Path

summary = {
    'response': None,
    'audit_log': None,
    'error': None,
}

try:
    orchestrator = SentifargoOrchestrator(llm_client=LLMStub(), memory_store=MemoryStore.from_env())
    response = orchestrator.handle('Hello there', user_id='notebook', use_rag=False)
    summary['response'] = {
        'route': response.get('route'),
        'answer_preview': (response.get('answer') or '')[:200],
        'latency_ms': response.get('latency_ms'),
    }
    print(summary['response'])
except Exception as exc:
    summary['error'] = str(exc)
    print('Orchestrator error:', exc)


In [ ]:
# Inspect latest audit log entry if present.
audit_dir = REPO_ROOT / 'logs' / 'audit'
if audit_dir.exists():
    files = sorted(audit_dir.glob('audit_*.jsonl'))
    latest = files[-1] if files else None
    if latest:
        lines = latest.read_text(encoding='utf-8', errors='ignore').splitlines()
        summary['audit_log'] = {
            'path': str(latest.relative_to(REPO_ROOT)),
            'entries': len(lines),
        }
        print('Audit log:', summary['audit_log'])
        if lines:
            print('Last entry:', lines[-1][:500])
else:
    print('Missing audit directory:', audit_dir)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_chat_orchestrator_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
